# 24 — Climate × Sheltered/Unsheltered: The Headline Analysis

**Hypothesis:** Cold climates push a larger share of homeless people into shelters without
reducing the *total* homeless rate. Warmer cities like LA, Phoenix, and Honolulu
have high unsheltered rates because sleeping outside is survivable year-round.

New York City: 'right to shelter' law + brutal winters → very low unsheltered % (~4%)
despite one of the highest total homeless rates in the US.

1. Scatter: unsheltered % vs January temperature (primary test)
2. Scatter: total homeless rate vs January temperature (does climate drive total rate?)
3. Faceted: unsheltered % vs temp, split by total homeless rate tier

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_city_data.csv')
df = df.dropna(subset=['homeless_rate_per_10k', 'unsheltered_pct', 'jan_temp_f'])
print(f'Loaded {len(df)} CoCs with climate data')

Loaded 101 CoCs with climate data


In [2]:
slope, intercept, r, p, se = stats.linregress(df['jan_temp_f'], df['unsheltered_pct'])
r2 = r ** 2
print(f'Jan temp vs unsheltered %: r={r:.3f}, R²={r2:.3f}, p={p:.4f}')
print(f'Each 10°F warmer → {slope*10:.1f}% more unsheltered')

x_range = np.linspace(df['jan_temp_f'].min(), df['jan_temp_f'].max(), 100)
y_range = slope * x_range + intercept

fig1 = px.scatter(
    df, x='jan_temp_f', y='unsheltered_pct',
    size='total_homeless',
    color='homeless_rate_per_10k', color_continuous_scale='Reds',
    text='city_name',
    hover_name='coc_name',
    hover_data={'lat': False, 'lon': False, 'state_postal': True, 'jan_temp_f': ':.0f', 'unsheltered_pct': ':.1f', 'total_homeless': ':,', 'homeless_rate_per_10k': ':.1f'},
    title=f'January Temperature vs. % Unsheltered Homeless (City/CoC Level)<br><sup>r={r:.3f}, R²={r2:.3f}, p={p:.4f} — warmer cities have higher unsheltered share; bubble size = total homeless; color = rate per 10k</sup>',
    labels={'jan_temp_f': 'Avg January Temperature (°F, NOAA 1991-2020)', 'unsheltered_pct': '% of Homeless Who Are Unsheltered'},
    size_max=60,
)
fig1.add_trace(go.Scatter(x=x_range, y=y_range, mode='lines', name='Regression', line=dict(color='black', width=2, dash='dash')))
fig1.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig1.show()

Jan temp vs unsheltered %: r=0.703, R²=0.494, p=0.0000
Each 10°F warmer → 7.0% more unsheltered


In [3]:
slope2, intercept2, r2v, p2, se2 = stats.linregress(df['jan_temp_f'], df['homeless_rate_per_10k'])
r2_2 = r2v ** 2
print(f'Jan temp vs total homeless rate: r={r2v:.3f}, R²={r2_2:.3f}, p={p2:.4f}')
print(f'Climate explains {r2_2*100:.0f}% of variance in total homeless rate')

x_range2 = np.linspace(df['jan_temp_f'].min(), df['jan_temp_f'].max(), 100)
y_range2 = slope2 * x_range2 + intercept2

fig2 = px.scatter(
    df, x='jan_temp_f', y='homeless_rate_per_10k',
    size='total_homeless',
    color='unsheltered_pct', color_continuous_scale='OrRd',
    text='city_name',
    hover_name='coc_name',
    title=f'January Temperature vs. Total Homeless Rate (City/CoC Level)<br><sup>r={r2v:.3f}, R²={r2_2:.3f}, p={p2:.4f} — color = % unsheltered</sup>',
    labels={'jan_temp_f': 'Avg January Temperature (°F)', 'homeless_rate_per_10k': 'Total Homeless Rate per 10k'},
    size_max=60,
)
fig2.add_trace(go.Scatter(x=x_range2, y=y_range2, mode='lines', name='Regression', line=dict(color='black', width=2, dash='dash')))
fig2.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig2.show()

Jan temp vs total homeless rate: r=0.181, R²=0.033, p=0.0700
Climate explains 3% of variance in total homeless rate


In [4]:
df['rate_tier'] = pd.qcut(df['homeless_rate_per_10k'], q=3, labels=['Low rate (bottom third)', 'Medium rate (middle third)', 'High rate (top third)'])

fig3 = px.scatter(
    df, x='jan_temp_f', y='unsheltered_pct',
    facet_col='rate_tier',
    color='homeless_rate_per_10k', color_continuous_scale='Reds',
    text='city_name',
    hover_name='coc_name',
    title='January Temperature vs. Unsheltered % — by Homeless Rate Tier<br><sup>Climate effect on shelter share holds across all rate tiers</sup>',
    labels={'jan_temp_f': 'Jan Temp (°F)', 'unsheltered_pct': '% Unsheltered'},
)
fig3.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig3.show()

In [5]:
warm = df[df['jan_temp_f'] >= 55]
cold = df[df['jan_temp_f'] <= 30]
print(f'Warm cities (Jan ≥ 55°F): mean unsheltered {warm["unsheltered_pct"].mean():.1f}%, mean rate {warm["homeless_rate_per_10k"].mean():.1f}/10k ({len(warm)} CoCs)')
print(f'Cold cities (Jan ≤ 30°F): mean unsheltered {cold["unsheltered_pct"].mean():.1f}%, mean rate {cold["homeless_rate_per_10k"].mean():.1f}/10k ({len(cold)} CoCs)')

Warm cities (Jan ≥ 55°F): mean unsheltered 36.8%, mean rate 41.0/10k (14 CoCs)
Cold cities (Jan ≤ 30°F): mean unsheltered 13.7%, mean rate 24.3/10k (35 CoCs)
